# Running ML Training and Serving Models on DKube

This notebook demonstrates how to submit an ML training program to the DKube platform and execute it. While this example covers basic usage, DKube’s API supports advanced configurations including resource specifications (GPUs, CPUs, memory) and node pinning.

Upon completion of the training, DKube automatically records all lineage information and versions the trained model. The trained model is then made available for serving.

Additionally, the notebook includes code for deploying the latest version of the trained model and testing it with sample data for inference.

In [ ]:
from dkube.sdk import *
from dkube.sdk.internal.dkube_api.rest import ApiException
import os
import random
import string

unique_name = "insurance-" + ''.join(random.choices(string.ascii_letters, k=6)) #Unique name across multiple runs
unique_name = unique_name.lower()
print(f"Unique name = {unique_name}. All the DKube resources in further cells will be created with this name")

#### Macros

In [ ]:
user = os.getenv('USERNAME') # Logged in user name

token = os.getenv('DKUBE_USER_ACCESS_TOKEN') # API key of the logged in user


dkube_resources = {
    "code": {"name": unique_name, "source": "https://github.com/oneconvergence/dkube-examples.git"},
    "dataset": {"name": unique_name, "source": "https://dkube-examples-data.s3.us-west-2.amazonaws.com/monitoring-insurance/training-data/insurance.csv"},
    "model": {"name": unique_name, "source": "dvs"}
}

dkube_runparams = {
    "training_framework": "tensorflow_2.6.0",
    "training_image": "ocdr/dkube-datascience-tf-cpu:v2.6.0-19",
    "training_script": "python insurance/training.py",
    "serving_image": "ocdr/tensorflowserver:2.6.0",
    "serving_transformer_script": "insurance/transformer.py"
}

#### DKube API Client

In [ ]:
# Dkube client handle
api = DkubeApi(token=token)
api.wait_interval = 20

#### Creating DKube resources required for ML Training

In [ ]:
print(f"Creating DKube code resource with name {unique_name}")
code = DkubeCode(user, name=dkube_resources['code']['name'])
code.update_git_details(dkube_resources['code']['source'], branch="training_v1")
api.create_code(code)

print(f"Creating DKube [dataset] resource with name {unique_name}")
dataset = DkubeDataset(user, name=dkube_resources['dataset']['name'])
dataset.update_puburl_details(dkube_resources['dataset']['source'], extract=False)
api.create_dataset(dataset)

print(f"Creating DKube [model] resource with name {unique_name}")
model = DkubeModel(user, name=dkube_resources['model']['name'])
model.update_model_source(source='dvs')
api.create_model(model)

#### Submit a ML training job to DKube

In [ ]:
training_name= unique_name

training = DkubeTraining(user, name=training_name, description='ML training for insurance data, triggered from sdk.')
training.update_container(framework=dkube_runparams['training_framework'], image_url=dkube_runparams['training_image'])
training.update_startupscript(dkube_runparams['training_script'])
training.add_code(dkube_resources['code']['name'])
training.add_input_dataset(dkube_resources['dataset']['name'], mountpath="/mnt/input")
training.add_output_model(dkube_resources['model']['name'], mountpath="/mnt/output")

api.create_training_run(training)

print("\n\n")
print("DKube Training run with name [{training_name}] is complete. Now model ready for serving.")

#### Deploy the trained model for inference

In [ ]:
serving_name = unique_name

serving = DkubeServing(user, name=serving_name, description='Serving for insurance model, triggered from sdk.')
serving.set_transformer(transformer=True, script=dkube_runparams['serving_transformer_script'])
serving.update_transformer_code(code=dkube_resources['code']['name'])
serving.update_transformer_image(image_url=dkube_runparams['training_image'])
serving.update_serving_model(dkube_resources['model']['name'])
serving.update_serving_image(image_url=dkube_runparams['serving_image'])
serving.update_autoscaling_config(min_replicas=1, max_concurrent_requests=10)

api.create_test_inference(serving)

print("\n\n")
print(f"DKube model deployment created with name [{serving_name}]. The deployment is ready for inference.")

#### Test the deployed model for inference

In [ ]:
s = api.get_test_inference(user, serving_name)
serving_url =s["job"]["parameters"]["generated"]["details"]["serving"]["servingurl"]

In [ ]:
import requests
import pandas as pd
import json

features = {'age':31,'sex':1,'bmi':20,'children':2,'smoker':1,'region':1}
df = pd.DataFrame(features, index=[0]) 

final_features = df.values

payload = {
        'signatures':{
            'inputs':[[{'data':df.to_csv(index=False)}]]
        },
        'instances': [],
        'token': token
    }
r = requests.post(serving_url, json=payload, headers = {'authorization': "Bearer " + token}, verify = False)
prediction = json.loads(r.content.decode('utf-8'))

print(f"Insurance premium amount for a person with features {str(features)} -> {prediction}$")